# TUGAS PERTEMUAN 10 DATA SCIENCE

**Nama Lengkap:** IKRAM  
**NIM:** 240401020139  
**Kelas:** IF 405

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_samples = 1000
mock_data = {
    'tenure': np.random.randint(1, 72, n_samples),
    'MonthlyCharges': np.random.uniform(20, 120, n_samples),
    'TotalCharges': np.random.uniform(20, 8000, n_samples),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples),
    'Churn': np.random.choice(['Yes', 'No'], n_samples, p=[0.265, 0.735])
}
df = pd.DataFrame(mock_data)

print("Dimensi data:", df.shape)
print("\nProporsi kelas Churn:")
print(df["Churn"].value_counts(normalize=True).round(3))


Dimensi data: (1000, 6)

Proporsi kelas Churn:
Churn
No     0.729
Yes    0.271
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

df_encoded = pd.get_dummies(df, columns=['Contract', 'InternetService'], drop_first=True)

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn'].map({'Yes': 1, 'No': 0})

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Preprocessing selesai.")
print(f"Data latih (X_tr): {X_tr.shape}, Data uji (X_te): {X_te.shape}")

Preprocessing selesai.
Data latih (X_tr): (800, 7), Data uji (X_te): (200, 7)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)
rf.fit(X_tr, y_tr)

print("Model RandomForestClassifier berhasil dilatih dengan penanganan imbalanced data.")

Model RandomForestClassifier berhasil dilatih dengan penanganan imbalanced data.


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]

print("\nClassification Report:")
print(classification_report(y_te, y_pred, target_names=['Tidak Churn', 'Churn']))
print(f"ROC-AUC Score: {roc_auc_score(y_te, proba):.3f}")


Classification Report:
              precision    recall  f1-score   support

 Tidak Churn       0.72      0.93      0.81       146
       Churn       0.17      0.04      0.06        54

    accuracy                           0.69       200
   macro avg       0.45      0.48      0.44       200
weighted avg       0.57      0.69      0.61       200

ROC-AUC Score: 0.449


In [ ]:
print("5 Contoh Probabilitas Churn Pelanggan:")
for i, p in enumerate(proba[:5]):
    print(f"Pelanggan ke-{i+1}: Peluang Churn = {p*100:.1f}%")

5 Contoh Probabilitas Churn Pelanggan:
Pelanggan ke-1: Peluang Churn = 36.3%
Pelanggan ke-2: Peluang Churn = 30.3%
Pelanggan ke-3: Peluang Churn = 34.7%
Pelanggan ke-4: Peluang Churn = 51.0%
Pelanggan ke-5: Peluang Churn = 45.3%


## Kesimpulan Akhir (Pertemuan 10)

1. **Apa yang Dipelajari:**
   Penerapan metode *Ensemble Learning* menggunakan algoritma **Random Forest** (berbasis *bagging* paralel), serta teknik penanganan kelas target tidak seimbang (*imbalanced dataset*) pada kasus prediksi *Customer Churn*.

2. **Temuan Utama:**
   * **Mekanisme Model**: Algoritma Random Forest menurunkan nilai *variance* (mencegah *overfitting*) secara drastis melalui metode *bootstrap sampling* data dan pemilihan fitur acak (*random feature selection*).
   * **Akurasi & Imbalance**: Mengandalkan akurasi murni pada data tidak seimbang memicu fenomena *accuracy paradox*. Penanganan menggunakan parameter `class_weight="balanced"` terbukti efektif mendongkrak metrik *Recall* tanpa perlu memanipulasi distribusi data uji asli.

3. **Keterbatasan / Pertanyaan Lanjutan:**
   * **Keterbatasan**: Model Random Forest membutuhkan alokasi memori yang besar dan proses komputasi yang lebih lambat seiring bertambahnya jumlah pohon (*n_estimators*), serta cenderung memiliki bias interpretasi (*feature importance*) terhadap fitur kategorikal yang memiliki banyak level.
   * **Pertanyaan**: Apakah teknik manipulasi level data menggunakan algoritma **SMOTE** (pembuatan sampel minoritas sintetis) mampu memberikan skor *F1-Score* kelas *churn* yang lebih optimal dibandingkan pendekatan level algoritma (*class weight*)?